In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F 
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-dataframe-drill").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 14:43:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill

The file sf-loan-performance-data-sample.csv contains a sample from Fannie Mae loan performance data https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/fannie-mae-single-family-loan-performance-data

The data dictionary is provided here https://capitalmarkets.fanniemae.com/media/6931/display

Our data scientists are interested in columns

- 2 Loan Identifier
- 20 Original Loan to Value Ratio (LTV)
- 31 Property State
- 33 Zip Code Short
- extract these columns, and select only loans in FL        
- save the results in csv format

## Dataset: Fannie Mae Single-Family Loan Performance

The file `data/sf-loan-performance-data-sample.csv` is a sample from the Fannie Mae dataset:  
https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/fannie-mae-single-family-loan-performance-data

Data dictionary: https://capitalmarkets.fanniemae.com/media/6931/display

The file uses **`|` as delimiter** and has **no header row** -- columns are positional (`_c0`, `_c1`, …).

Key columns (1-indexed per data dictionary):
- Column 2 (`_c1`): Loan Identifier
- Column 20 (`_c19`): Original Loan to Value Ratio (LTV)
- Column 31 (`_c30`): Property State
- Column 33 (`_c32`): Zip Code (short)

In [3]:
file_name = os.path.join(os.getcwd(), "data/sf-loan-performance-data-sample.csv")
data = ( spark.read.option("header", "false")
            .option("multiLine", "false")
            .option("inferSchema", "false")
            .option("delimiter", "|")
            .csv(file_name) )
data

DataFrame[_c0: string, _c1: string, _c2: string, _c3: string, _c4: string, _c5: string, _c6: string, _c7: string, _c8: string, _c9: string, _c10: string, _c11: string, _c12: string, _c13: string, _c14: string, _c15: string, _c16: string, _c17: string, _c18: string, _c19: string, _c20: string, _c21: string, _c22: string, _c23: string, _c24: string, _c25: string, _c26: string, _c27: string, _c28: string, _c29: string, _c30: string, _c31: string, _c32: string, _c33: string, _c34: string, _c35: string, _c36: string, _c37: string, _c38: string, _c39: string, _c40: string, _c41: string, _c42: string, _c43: string, _c44: string, _c45: string, _c46: string, _c47: string, _c48: string, _c49: string, _c50: string, _c51: string, _c52: string, _c53: string, _c54: string, _c55: string, _c56: string, _c57: string, _c58: string, _c59: string, _c60: string, _c61: string, _c62: string, _c63: string, _c64: string, _c65: string, _c66: string, _c67: string, _c68: string, _c69: string, _c70: string, _c71: 

In [4]:
data.show(3)

26/06/11 14:45:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+------------+------+---+-----+-----+----+-----+-----+--------+----+----+----+------+------+----+----+----+------+----+----+----+----+----+----+----+----+----+----+----+----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+
| _c0|         _c1|   _c2|_c3|  _c4|  _c5| _c6|  _c7|  _c8|     _c9|_c10|_c11|_c12|  _c13|  _c14|_c15|_c16|_c17|  _c18|_c19|_c20|_c21|_c22|_c23|_c24|_c25|_c26|_c27|_c28|_c29|_c30| _c31|_c32|_c33|_c34|_c35|_c36|_c37|_c38|_c39|_c40|_c41|_c42|_c43|_c44|_c45|_c46|_c47|_c48|_c49|_c50|_c51|_c52|_c53|_c54|_c55|_c56|_c57|_c58|_c59|_c60|_c61|_c62|_c63|_c64|_c65|_c66|_c67|_c68|_c69|_c70|_c71|_c72|_c73|_c74|_c75|_c76|_c77|_c78|_c79|_

In [5]:
data.first()

Row(_c0=None, _c1='100023020488', _c2='082009', _c3='R', _c4='Other', _c5='Other', _c6=None, _c7='5.375', _c8='5.375', _c9='55000.00', _c10=None, _c11='0.00', _c12='240', _c13='082009', _c14='102009', _c15='-1', _c16='241', _c17='240', _c18='092029', _c19='55', _c20='55', _c21='1', _c22='36', _c23='714', _c24=None, _c25='N', _c26='C', _c27='SF', _c28='1', _c29='P', _c30='OH', _c31='17140', _c32='452', _c33=None, _c34='FRM', _c35='N', _c36='N', _c37=None, _c38=None, _c39='00', _c40=None, _c41='N', _c42=None, _c43=None, _c44=None, _c45=None, _c46=None, _c47=None, _c48=None, _c49=None, _c50=None, _c51=None, _c52=None, _c53=None, _c54=None, _c55=None, _c56=None, _c57=None, _c58=None, _c59=None, _c60=None, _c61=None, _c62=None, _c63=None, _c64=None, _c65=None, _c66=None, _c67=None, _c68=None, _c69=None, _c70=None, _c71=None, _c72=None, _c73='N', _c74=None, _c75=None, _c76=None, _c77=None, _c78=None, _c79=None, _c80='N', _c81=None, _c82=None, _c83=None, _c84=None, _c85=None, _c86='N', _c87=N

In [6]:
step1 = data.select(F.col("_c1"), F.col("_c19"), F.col("_c30"), F.col("_c32"))
step2 = step1.where(F.col("_c30") == "FL")
step2.write.option("header", "false").csv(os.path.join(os.getcwd(), "tmp/sf-loan-performance-data-out.csv"))

In [12]:
spark.stop()